In [1]:
import numpy as np
from numba import cuda

# -------------------------------
# Kernel: One reduction step
# -------------------------------
@cuda.jit
def reduce_step(arr, stride, n):
    i = cuda.grid(1)
    
    idx = 2 * stride * i
    
    if idx + stride < n:
        arr[idx] += arr[idx + stride]


# -------------------------------
# Main execution
# -------------------------------
def parallel_reduction_sum(arr):
    n = arr.size

    # Copy to device
    d_arr = cuda.to_device(arr)

    threads_per_block = 256
    stride = 1

    # Log2(N) reduction steps
    while stride < n:
        blocks = (n // (2 * stride) + threads_per_block - 1) // threads_per_block
        
        reduce_step[blocks, threads_per_block](d_arr, stride, n)
        
        stride *= 2

    # Copy result back
    result = d_arr.copy_to_host()
    
    return result[0]


# -------------------------------
# Test
# -------------------------------
if __name__ == "__main__":
    N = 2048  # works best for power of 2

    arr = np.arange(1, N + 1, dtype=np.float32)

    gpu_sum = parallel_reduction_sum(arr)
    cpu_sum = np.sum(arr)

    print("GPU Sum:", gpu_sum)
    print("CPU Sum:", cpu_sum)

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


GPU Sum: 2098176.0
CPU Sum: 2098176.0


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
